# Polars → HF Buckets: Streaming Parquet Sink Demo

This notebook demonstrates a proof-of-concept streaming parquet sink from Polars to HuggingFace Buckets via the XET protocol.

**What it does**: `sink_parquet("hf://buckets/...")` streams parquet data directly to an HF Bucket with constant memory usage — O(row_group_size), not O(dataset_size).

**Prerequisites**:
- A HuggingFace account with a [Bucket](https://huggingface.co/new-bucket) created
- An HF token with write access (add it to Colab Secrets as `HF_TOKEN`)
- The custom Polars wheels (built from the `feature/hf-bucket-sink` branch)

## 1. Install Custom Wheels

Download the wheels from CI artifacts and install them.

**Important**: Use `--no-deps --force-reinstall` to prevent pip from replacing the custom `polars-runtime-32` wheel with the upstream PyPI version (they share the same version number).

After installing, **restart the runtime** (Runtime → Restart runtime) before continuing.

In [ ]:
# Upload the two wheel files to Colab first (use the file browser or upload API),
# then run this cell.
#
# Wheels are built by the "Build HF Sink Wheels" GitHub Actions workflow:
#   https://github.com/davanstrien/polars/actions/workflows/build-hf-sink-wheels.yml
#
# Download the x86_64 artifacts: polars-*.whl and polars_runtime_32-*.whl

!pip uninstall polars polars-runtime-32 -y -q
!pip install --no-deps --force-reinstall polars-*.whl polars_runtime_32-*.whl -q

**⚠️ Restart the runtime now** (Runtime → Restart runtime), then continue from cell 2.

## 2. Setup

In [ ]:
import polars as pl
import os

# Load HF_TOKEN from Colab Secrets
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print(f"Polars version: {pl.__version__}")
print(f"HF token loaded: {'HF_TOKEN' in os.environ}")

In [ ]:
# ── Configure your bucket here ──
# Create a bucket at https://huggingface.co/new-bucket

NAMESPACE = "<YOUR_NAMESPACE>"  # your HF username or org
BUCKET = "<YOUR_BUCKET>"        # bucket name

BUCKET_BASE = f"hf://buckets/{NAMESPACE}/{BUCKET}"

## 3. Example 1: Simple Write

The simplest case — create a DataFrame and sink it directly to a bucket.

In [ ]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "score": [0.95, 0.87, 0.92],
})

df.lazy().sink_parquet(
    f"{BUCKET_BASE}/simple-test.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]},
)

print("Written to bucket.")

In [ ]:
# Read back and verify the data round-trips correctly
from huggingface_hub import download_bucket_files
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    download_bucket_files(
        f"{NAMESPACE}/{BUCKET}",
        files=[("simple-test.parquet", f"{tmpdir}/simple-test.parquet")],
        token=os.environ["HF_TOKEN"],
    )
    result = pl.read_parquet(f"{tmpdir}/simple-test.parquet")

print(result)

(
    pl.scan_parquet(
        "hf://datasets/wikimedia/wikipedia/20231101.en/train-00000-of-00041.parquet",
        storage_options={"token": os.environ["HF_TOKEN"]},
    )
    .filter(pl.col("text").str.len_chars() > 5000)
    .select("id", "url", "title", "text")
    .sink_parquet(
        f"{BUCKET_BASE}/wikipedia-long-articles.parquet",
        storage_options={"token": os.environ["HF_TOKEN"]},
    )
)

print("Scan -> filter -> sink complete.")

In [ ]:
(
    pl.scan_parquet(
        "hf://datasets/wikimedia/wikipedia/20231101.en/train-00000-of-00041.parquet"
    )
    .filter(pl.col("text").str.len_chars() > 5000)
    .select("id", "url", "title", "text")
    .sink_parquet(f"{BUCKET_BASE}/wikipedia-long-articles.parquet")
)

print("Scan → filter → sink complete.")

import time

n_rows = 100_000

df = pl.DataFrame({
    "id": pl.arange(0, n_rows, eager=True),
    "value": pl.arange(0, n_rows, eager=True).cast(pl.Float64) * 0.001,
    "category": pl.arange(0, n_rows, eager=True) % 100,
    "text": pl.Series([f"row-{i}-" + "x" * 200 for i in range(n_rows)]),
})

start = time.time()
df.lazy().sink_parquet(
    f"{BUCKET_BASE}/scale-test-{n_rows}.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]},
)
elapsed = time.time() - start

print(f"{n_rows:,} rows written in {elapsed:.1f}s")

---

## Notes

- **This is a PoC** — single-file output, requires custom wheels built from the feature branch.
- **Token refresh**: XET upload tokens are automatically refreshed for long-running uploads.
- **Memory model**: O(row_group_size), not O(dataset_size). Parquet bytes stream through a bounded channel to the XET upload. RSS stays constant.
- **Branch**: [`feature/hf-bucket-sink`](https://github.com/davanstrien/polars/tree/feature/hf-bucket-sink)
- **Core diff**: ~52 lines in 7 polars-stream/polars-io files, all `#[cfg(feature = "hf_bucket_sink")]` gated.
- **E2E tests**: `py-polars/tests/unit/io/cloud/test_hf_bucket_sink.py` — smoke, 10K, and 10M row tests (gated behind `HF_TOKEN` + `pytest.mark.slow`).
- **Read support**: `pl.read_parquet("hf://buckets/...")` is not yet supported (separate concern).

See [polars-hf-bucket-sink-demo.md](./polars-hf-bucket-sink-demo.md) for the full technical write-up.

---

## Notes

- **This is a PoC** — single-file output, no token refresh for long uploads, requires custom wheels.
- **Memory model**: O(row_group_size), not O(dataset_size). Parquet bytes stream through a bounded channel to the XET upload. RSS stays constant.
- **Branch**: [`feature/hf-bucket-sink`](https://github.com/davanstrien/polars/tree/feature/hf-bucket-sink)
- **Core diff**: ~52 lines in 7 polars-stream/polars-io files, all `#[cfg(feature = "hf_bucket_sink")]` gated.
- **Read support**: `pl.read_parquet("hf://buckets/...")` is not yet supported (separate concern).

See [polars-hf-bucket-sink-demo.md](./polars-hf-bucket-sink-demo.md) for the full technical write-up.